# SVD — Revisão v5 para *Journal of Voice*
### Reestruturado a partir dos comentários do Revisor #1 (JVOICE-D-26-00653)

**Pergunta central única:** sob avaliação speaker-independente, features acústicas
clinicamente validadas (Praat) discriminam voz patológica de saudável no SVD — e
representações auto-supervisionadas (wav2vec 2.0) oferecem vantagem?

**Framework primário:** classificação binária (saudável vs. patológico).
**Papel da estatística:** (1) validar clinicamente as features *antes* do ML;
(2) verificar via SHAP se o modelo aprendeu esse mesmo sinal.

| Crítica do revisor | Correção nesta versão |
|---|---|
| Frameworks desconectados | Estatística subordinada ao ML; ponte via SHAP (Bloco 7) |
| Binário vs. multiclasse | Tudo binário; multiclasse → Suplementar |
| Cohen's d inadequado | Substituído por r rank-biserial |
| Sem nested CV | GridSearchCV interno em 3 folds (Bloco 4) |
| Leakage no PCA/normalização | Ambos dentro do Pipeline sklearn |
| wav2vec sem fine-tuning | Ambos os regimes avaliados (Bloco 6) |
| Inclusão/exclusão obscura | Auditoria completa (Bloco 1) |
| 12,24 gravações/speaker | Distribuição documentada (Bloco 1) |
| Parâmetros não reportados | Dicionário `P` explícito (Bloco 2) |
| F0 → fo | Notação Titze et al. (2015) |
| Figuras de baixa qualidade | TIFF 600 dpi, paleta acessível |
| ROCs separadas | Painel único (Fig. 2) |


---
### Correções da 2ª revisão (JVOICE-D-26-01691) aplicadas nesta cópia

| Problema encontrado | Correção |
|---|---|
| `speaker_id = diagnóstico_sessão`: a mesma sessão em duas pastas virava dois "speakers", com as 12 gravações duplicadas e podendo cair uma cópia no treino e outra no teste | Cada sessão entra uma vez só (diagnóstico por prioridade: lesão orgânica antes de disfonia funcional) |
| A pasta de gravação é a **sessão**, não o falante; 78 falantes têm mais de uma sessão | Partições agrupadas pelo ID real do falante (`summary.csv` oficial da SVD) |
| Sexo e idade "não disponíveis" | Lidos do `summary.csv`; idade na data da gravação |
| `jitter_local` (fração) é cópia exata de `jitter_local_pct` | Removido: 78 features, como diz o artigo |
| Imputação pela mediana do corpus inteiro antes do CV | Imputação dentro de cada pipeline, só com o treino |
| Estatística do Bloco 3 por gravação | Por sessão (mediana das produções), como descrito no artigo; fo DP só no pitch normal |
| Bootstrap sobre as 5 AUCs de dobra | Bootstrap pareado de falantes sobre as previsões fora da amostra |

Grades, arquiteturas e sementes originais foram mantidas. A busca do wav2vec, a grade do FT-Transformer e as ablações
descritas no artigo estão no notebook `svd_tabelas_5a7_corrigidas.ipynb`.


## Bloco 1 — Setup e auditoria de inclusão/exclusão

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 1 — SETUP, INCLUSÃO/EXCLUSÃO DOCUMENTADA, CONVERSÃO
# Responde: R1-Methods (inclusão/exclusão), R1-Table1 (12.24 rec/speaker)
# ══════════════════════════════════════════════════════════════
import os, glob, subprocess, time, threading, warnings, re, json
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

os.system('pip install praat-parselmouth pingouin umap-learn soundfile sbvoicedb -q')
os.system('apt-get install -y sox libsox-fmt-all -q')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR    = '/content/drive/MyDrive/svd_data'
WAV_DIR      = '/content/svd_wav'
RANDOM_STATE = 42
N_FOLDS      = 5
N_INNER      = 3          # nested CV interno para hiperparâmetros
SAMPLE_RATE  = 16000

# ─── FIGURAS: QUALIDADE DE PUBLICAÇÃO ─────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np, pandas as pd
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.edgecolor':'#000000', 'axes.linewidth':0.8,
    'axes.labelcolor':'#000000', 'text.color':'#000000',
    'xtick.color':'#000000', 'ytick.color':'#000000',
    'grid.color':'#CCCCCC', 'grid.linewidth':0.5, 'grid.alpha':0.6,
    'font.family':'sans-serif', 'font.sans-serif':['Arial','DejaVu Sans'],
    'font.size':10, 'axes.titlesize':11, 'axes.labelsize':10,
    'xtick.labelsize':9, 'ytick.labelsize':9, 'legend.fontsize':9,
    'figure.dpi':100, 'savefig.dpi':600,          # 600 dpi = padrão Elsevier
    'savefig.bbox':'tight', 'savefig.facecolor':'white',
    'savefig.format':'tiff',
})
# Paleta segura para daltonismo e impressão P&B
PAL = ['#0072B2','#D55E00','#009E73','#CC79A7','#E69F00','#56B4E9']

DIAGNOSES = {
    'healthy':                   0,
    'hyperfunctional_dysphonia': 1,
    'laryngitis':                1,
    'nodule':                    1,
    'polyp':                     1,
    'vocal_fold_paralysis':      1,
}
PITCH_MAP = {'n':'normal','h':'high','l':'low','lhl':'rise_fall'}

# ─── COLETA COM LOG DE INCLUSÃO/EXCLUSÃO ──────────────────────
audit = {'speakers_scanned':0, 'speakers_no_vowels_dir':0,
         'speakers_included':0, 'files_scanned':0,
         'files_bad_name':0, 'files_included':0,
         'speakers_by_n_recordings':{}}

def collect(root):
    recs = []
    for diag in sorted(os.listdir(root)):
        if diag not in DIAGNOSES: continue
        dpath = os.path.join(root, diag)
        if not os.path.isdir(dpath): continue
        for spk in sorted(os.listdir(dpath)):
            spath = os.path.join(dpath, spk)
            if not os.path.isdir(spath): continue
            audit['speakers_scanned'] += 1
            vpath = os.path.join(spath,'vowels')
            if not os.path.isdir(vpath):
                audit['speakers_no_vowels_dir'] += 1
                continue
            n_this = 0
            for f in sorted(os.listdir(vpath)):
                if not f.endswith('.nsp'): continue
                audit['files_scanned'] += 1
                m = re.match(r'^(\d+)-([aeiou]+)_([a-z]+)\.nsp$', f)
                if not m:
                    audit['files_bad_name'] += 1
                    continue
                sid, vowel, pcode = m.group(1), m.group(2), m.group(3)
                recs.append({
                    'nsp': os.path.join(vpath,f),
                    'speaker_id': f"{diag}_{sid}",   # ID único global
                    'session_id': sid,
                    'diagnosis': diag,
                    'label': DIAGNOSES[diag],
                    'vowel': vowel,
                    'pitch': PITCH_MAP.get(pcode,pcode),
                })
                audit['files_included'] += 1
                n_this += 1
            if n_this > 0:
                audit['speakers_included'] += 1
                audit['speakers_by_n_recordings'][n_this] = \
                    audit['speakers_by_n_recordings'].get(n_this,0) + 1
    return recs

records = collect(DRIVE_DIR)

print('═'*66)
print('AUDITORIA DE INCLUSÃO/EXCLUSÃO  (para Methods do manuscrito)')
print('═'*66)
print(f"Pastas de speaker varridas ..... {audit['speakers_scanned']}")
print(f"  sem diretório /vowels ........ {audit['speakers_no_vowels_dir']}")
print(f"  incluídos .................... {audit['speakers_included']}")
print(f"Arquivos .nsp varridos ......... {audit['files_scanned']}")
print(f"  nome fora do padrão .......... {audit['files_bad_name']}")
print(f"  incluídos .................... {audit['files_included']}")
print('\nDISTRIBUIÇÃO DE GRAVAÇÕES POR SPEAKER:')
for n in sorted(audit['speakers_by_n_recordings']):
    print(f"  {n:2d} gravações: {audit['speakers_by_n_recordings'][n]:4d} speakers")
mean_rec = audit['files_included']/max(audit['speakers_included'],1)
print(f"\nMédia = {mean_rec:.2f} gravações/speaker")
print('→ Speakers com >12 gravações têm sessões repetidas no SVD')
print('  (múltiplas visitas clínicas registradas sob o mesmo ID de sessão).')
print('  Isto explica a média >12 apontada pelo revisor.')

with open('/content/audit_inclusion.json','w') as f:
    json.dump(audit,f,indent=2)

for d in sorted(DIAGNOSES):
    r = [x for x in records if x['diagnosis']==d]
    print(f"  {d}: {len(r)} gravações, {len(set(x['speaker_id'] for x in r))} speakers")

# ─── CONVERSÃO NSP → WAV ──────────────────────────────────────
os.makedirs(WAV_DIR, exist_ok=True)
def conv(rec):
    stem  = os.path.splitext(os.path.basename(rec['nsp']))[0]
    out_d = os.path.join(WAV_DIR, rec['diagnosis'], rec['session_id'])
    os.makedirs(out_d, exist_ok=True)
    out_f = os.path.join(out_d, f'{stem}.wav')
    if os.path.exists(out_f): return 'skip'
    r = subprocess.run(['sox',rec['nsp'],'-r',str(SAMPLE_RATE),'-c','1',out_f],
                       capture_output=True)
    if r.returncode==0: return 'ok'
    try:
        import soundfile as sf, librosa as lb
        raw = np.fromfile(rec['nsp'],dtype=np.int16)[256:]
        a   = lb.resample(raw.astype(np.float32)/32768.,orig_sr=50000,target_sr=SAMPLE_RATE)
        sf.write(out_f,a,SAMPLE_RATE); return 'ok'
    except Exception: return 'fail'

print(f'\nConvertendo {len(records)} NSPs...')
lk=threading.Lock(); ok=sk=fl=0; t0=time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    fut={ex.submit(conv,r):r for r in records}
    for i,f in enumerate(as_completed(fut),1):
        s=f.result()
        with lk:
            ok+= s=='ok'; sk+= s=='skip'; fl+= s=='fail'
        if i%200==0 or i==len(records):
            e=time.time()-t0; eta=(len(records)-i)/(i/e)
            print(f'\r  {i}/{len(records)} ✅{ok} ⏭{sk} ❌{fl} ETA {int(eta//60)}m{int(eta%60):02d}s',end='')
print(f'\n✅ {ok} convertidos, {sk} em cache, {fl} falhas')

for r in records:
    stem = os.path.splitext(os.path.basename(r['nsp']))[0]
    r['wav'] = os.path.join(WAV_DIR,r['diagnosis'],r['session_id'],f'{stem}.wav')
records = [r for r in records if os.path.exists(r['wav'])]
print(f'Registros válidos: {len(records)}')
print('✅ Bloco 1 completo')


## Bloco 2 — Extração de features (parâmetros explícitos)

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 2 — EXTRAÇÃO DE FEATURES
# Responde: R1 "parâmetros de extração explícitos", "agregação
#           frame→gravação", "citar parselmouth/librosa", "fo não F0"
# ══════════════════════════════════════════════════════════════
from tqdm.auto import tqdm
import parselmouth
from parselmouth.praat import call
import librosa
from sklearn.model_selection import StratifiedGroupKFold

CACHE = '/content/features_v5.pkl'
if not Path(CACHE).exists() and Path(f'{DRIVE_DIR}/features_v5.pkl').exists():
    CACHE = f'{DRIVE_DIR}/features_v5.pkl'      # cópia persistida no Drive

# ─── PARÂMETROS DE EXTRAÇÃO — TODOS EXPLÍCITOS ────────────────
# (reportar integralmente no manuscrito, seção Methods)
P = dict(
    # Praat — Pitch (autocorrelation)
    pitch_floor      = 75.0,   # Hz  — padrão Praat para voz adulta
    pitch_ceiling    = 600.0,  # Hz
    time_step        = 0.0,    # 0 = automático (= 0.75/pitch_floor)
    # Praat — Jitter/Shimmer
    period_floor     = 0.0001, # s
    period_ceiling   = 0.02,   # s
    max_period_factor= 1.3,
    max_ampl_factor  = 1.6,
    # Praat — Harmonicity (cross-correlation)
    hnr_time_step    = 0.01,   # s
    hnr_min_pitch    = 75.0,   # Hz
    hnr_silence_thr  = 0.1,
    hnr_periods_win  = 1.0,
    # CPPS (Hillenbrand) — via Praat PowerCepstrogram
    cpps_pitch_floor = 60.0,   # Hz
    cpps_pitch_ceil  = 330.0,  # Hz
    cpps_tolerance   = 0.05,
    # librosa — MFCC / espectrais
    n_mfcc           = 13,
    n_mels           = 40,
    frame_ms         = 25,
    hop_ms           = 10,
    n_fft            = 512,
)
print('Parâmetros de extração (reportar em Methods):')
for k,v in P.items(): print(f'  {k:<18} = {v}')

def praat_features(wav):
    """Perturbação e qualidade via Praat (parselmouth).
    Agregação: cada medida é um valor único por gravação,
    computado pelo Praat sobre toda a gravação (não frame-a-frame).
    """
    try:
        snd = parselmouth.Sound(wav)
        pit = call(snd,"To Pitch", P['time_step'], P['pitch_floor'], P['pitch_ceiling'])
        pp  = call([snd,pit],"To PointProcess (cc)")
        hnr = call(snd,"To Harmonicity (cc)", P['hnr_time_step'],
                   P['hnr_min_pitch'], P['hnr_silence_thr'], P['hnr_periods_win'])
        # CPPS via PowerCepstrogram (implementação Praat de Hillenbrand)
        try:
            pc   = call(snd,"To PowerCepstrogram", P['cpps_pitch_floor'], 0.002,
                        5000.0, 50.0)
            cpps = call(pc,"Get CPPS", False, 0.02, 0.0005,
                        P['cpps_pitch_floor'], P['cpps_pitch_ceil'],
                        P['cpps_tolerance'], "Parabolic", 0.001, 0.05,
                        "Straight", "Robust")
        except Exception:
            cpps = np.nan
        a,b,c,d = 0,0,P['period_floor'],P['period_ceiling']
        e,f     = P['max_period_factor'], P['max_ampl_factor']
        return {
          'jitter_local' : call(pp,"Get jitter (local)",a,b,c,d,e),
          'jitter_local_pct': call(pp,"Get jitter (local)",a,b,c,d,e)*100,
          'jitter_rap'   : call(pp,"Get jitter (rap)",a,b,c,d,e),
          'jitter_ppq5'  : call(pp,"Get jitter (ppq5)",a,b,c,d,e),
          'shimmer_local': call([snd,pp],"Get shimmer (local)",a,b,c,d,e,f),
          'shimmer_dB'   : call([snd,pp],"Get shimmer (local_dB)",a,b,c,d,e,f),
          'shimmer_apq3' : call([snd,pp],"Get shimmer (apq3)",a,b,c,d,e,f),
          'shimmer_apq5' : call([snd,pp],"Get shimmer (apq5)",a,b,c,d,e,f),
          'hnr_dB'       : call(hnr,"Get mean",0,0),
          'cpps_dB'      : cpps,
          'fo_mean'      : call(pit,"Get mean",0,0,"Hertz"),
          'fo_sd'        : call(pit,"Get standard deviation",0,0,"Hertz"),
          'fo_min'       : call(pit,"Get minimum",0,0,"Hertz","Parabolic"),
          'fo_max'       : call(pit,"Get maximum",0,0,"Hertz","Parabolic"),
          'voiced_frac'  : call(pit,"Count voiced frames")/max(call(pit,"Get number of frames"),1),
        }
    except Exception:
        return {k:np.nan for k in ['jitter_local','jitter_local_pct','jitter_rap',
            'jitter_ppq5','shimmer_local','shimmer_dB','shimmer_apq3','shimmer_apq5',
            'hnr_dB','cpps_dB','fo_mean','fo_sd','fo_min','fo_max','voiced_frac']}

def spectral_features(audio, sr):
    """Espectrais/cepstrais via librosa.
    AGREGAÇÃO FRAME→GRAVAÇÃO: média e desvio-padrão ao longo dos frames
    (explicitado no manuscrito conforme solicitado pelo revisor).
    """
    hop   = int(P['hop_ms']/1000*sr)
    frame = int(P['frame_ms']/1000*sr)
    S     = np.abs(librosa.stft(audio, n_fft=P['n_fft'], hop_length=hop,
                                win_length=frame))
    o = {}
    for nm, fn in [('centroid', librosa.feature.spectral_centroid),
                   ('bandwidth',librosa.feature.spectral_bandwidth),
                   ('rolloff',  librosa.feature.spectral_rolloff)]:
        v = fn(S=S, sr=sr)[0]
        o[f'spec_{nm}_mean']=float(np.mean(v)); o[f'spec_{nm}_sd']=float(np.std(v))
    fl = librosa.feature.spectral_flatness(S=S)[0]
    o['spec_flatness_mean']=float(np.mean(fl)); o['spec_flatness_sd']=float(np.std(fl))
    rms= librosa.feature.rms(y=audio, frame_length=frame, hop_length=hop)[0]
    o['rms_mean']=float(np.mean(rms)); o['rms_sd']=float(np.std(rms))
    z  = librosa.feature.zero_crossing_rate(audio, frame_length=frame, hop_length=hop)[0]
    o['zcr_mean']=float(np.mean(z));   o['zcr_sd']=float(np.std(z))
    mf = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=P['n_mfcc'],
                              n_mels=P['n_mels'], hop_length=hop, n_fft=P['n_fft'])
    dm = librosa.feature.delta(mf); d2 = librosa.feature.delta(mf, order=2)
    for i in range(P['n_mfcc']):
        o[f'mfcc{i+1}_mean']  = float(np.mean(mf[i]))
        o[f'mfcc{i+1}_sd']    = float(np.std(mf[i]))
        o[f'dmfcc{i+1}_mean'] = float(np.mean(dm[i]))
        o[f'd2mfcc{i+1}_mean']= float(np.mean(d2[i]))
    return o

def extract(wav, sr=SAMPLE_RATE):
    try:
        a,_ = librosa.load(wav, sr=sr, mono=True)
        a   = librosa.util.normalize(a)
        if len(a) < sr*0.2: return None
        d = praat_features(wav); d.update(spectral_features(a,sr)); return d
    except Exception:
        return None

if Path(CACHE).exists():
    feat = pd.read_pickle(CACHE); print(f'✅ Cache: {feat.shape}')
else:
    rows, bad = [], 0
    for r in tqdm(records, desc='Extraindo'):
        d = extract(r['wav'])
        if d is None: bad += 1; continue
        d.update({k:r[k] for k in ['speaker_id','session_id','label',
                                    'diagnosis','vowel','pitch','wav']})
        rows.append(d)
    feat = pd.DataFrame(rows); feat.to_pickle(CACHE)
    print(f'✅ {len(feat)} extraídas, {bad} falhas')

META = ['speaker_id','session_id','label','diagnosis','vowel','pitch','wav']

# ══════════════ CORREÇÕES DA 2ª REVISÃO ══════════════
import sbvoicedb
_S = pd.read_csv(os.path.join(os.path.dirname(sbvoicedb.__file__), 'summary.csv'))
_S['age'] = (pd.to_datetime(_S.AufnahmeDatum) - pd.to_datetime(_S.Geburtsdatum)).dt.days / 365.25
_S = _S.set_index('AufnahmeID')
feat['svd_session'] = feat['session_id'].astype(int)
feat['svd_speaker'] = feat.svd_session.map(_S['SprecherID'])
feat['sex']         = feat.svd_session.map(_S['Geschlecht']).map({'w': 'F', 'm': 'M'})
feat['age']         = feat.svd_session.map(_S['age'])
assert feat.svd_speaker.notna().all(), 'sessão sem metadados no summary.csv'

# (a) auditoria: com o agrupamento antigo (speaker_id = diagnóstico_sessão), quantas cópias caíam em treino e teste
_nd  = feat.groupby('svd_session').diagnosis.nunique()
DUP  = _nd[_nd > 1].index
_old = np.full(len(feat), -1)
for k, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
                            .split(feat, feat.label, feat.speaker_id)):
    _old[te] = k
_split = feat[feat.svd_session.isin(DUP)].assign(f=_old[feat.svd_session.isin(DUP).values]) \
            .groupby('svd_session').f.nunique()
print(f'Sessões em duas pastas: {len(DUP)} | gravações duplicadas: {feat.duplicated(["svd_session","vowel","pitch"]).sum()}')
print(f'  com o agrupamento antigo, {int((_split > 1).sum())} dessas sessões tinham cópias em dobras diferentes '
      f'(mesma gravação no treino e no teste)')

# (b) uma cópia por sessão: diagnóstico por prioridade (lesão orgânica antes de disfonia funcional)
PRIORITY = ['vocal_fold_paralysis', 'polyp', 'nodule', 'laryngitis', 'hyperfunctional_dysphonia']
_p   = feat.diagnosis.map({d: i for i, d in enumerate(PRIORITY)}).fillna(len(PRIORITY))
_min = _p.groupby(feat.svd_session).transform('min')
feat = feat[~(feat.svd_session.isin(DUP) & (_p != _min))] \
           .drop_duplicates(['svd_session', 'vowel', 'pitch']).reset_index(drop=True)

# (c) jitter_local (fração) = jitter_local_pct / 100 → fica só a versão em %
META += ['svd_session', 'svd_speaker', 'sex', 'age']
FEATS = [c for c in feat.columns if c not in META and c != 'jitter_local']

# (d) sem imputação global: NaN ficam e são imputados dentro de cada pipeline, só com o treino
n_nan = int(feat[FEATS].isna().sum().sum())
print(f'\nFeatures: {len(FEATS)} | NaN (imputados dentro do CV): {n_nan}')
print(f'Gravações: {len(feat)} | Sessões: {feat.svd_session.nunique()} | Falantes reais: {feat.svd_speaker.nunique()}')
print(feat.groupby('diagnosis').agg(gravacoes=('label','size'), sessoes=('svd_session','nunique'),
                                    falantes=('svd_speaker','nunique'),
                                    mulheres_pct=('sex', lambda s: round(100*(s=='F').mean(),1)),
                                    idade_media=('age', lambda a: round(a.mean(),1)),
                                    idade_dp=('age', lambda a: round(a.std(),1))).to_string())
print('✅ Bloco 2 completo')


## Bloco 3 — Validação clínica binária (rank-biserial)

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 3 — VALIDAÇÃO CLÍNICA DOS FEATURES (framework BINÁRIO)
# Responde: R1 "binário vs multiclasse", "Cohen's d inadequado",
#           "todas as features, não subconjunto"
#
# PAPEL NO ARTIGO: esta análise NÃO é um fim em si. Ela responde
# a uma pergunta específica que antecede o ML:
#   "As features Praat capturam diferença clínica entre voz
#    saudável e patológica neste corpus?"
# O resultado justifica o conjunto de features levado ao modelo.
# ══════════════════════════════════════════════════════════════
from scipy.stats import mannwhitneyu, shapiro

# Features clínicas interpretáveis (Praat) — o revisor pediu clareza
# sobre por que um subconjunto: são as medidas com norma clínica publicada
CLIN = ['jitter_local_pct','jitter_rap','jitter_ppq5',
        'shimmer_local','shimmer_dB','shimmer_apq3','shimmer_apq5',
        'hnr_dB','cpps_dB','fo_mean','fo_sd','voiced_frac']
LBL = {'jitter_local_pct':'Jitter local (%)','jitter_rap':'Jitter RAP',
       'jitter_ppq5':'Jitter PPQ5','shimmer_local':'Shimmer local',
       'shimmer_dB':'Shimmer (dB)','shimmer_apq3':'Shimmer APQ3',
       'shimmer_apq5':'Shimmer APQ5','hnr_dB':'HNR (dB)',
       'cpps_dB':'CPPS (dB)','fo_mean':'fo médio (Hz)',
       'fo_sd':'fo DP (Hz)','voiced_frac':'Fração sonorizada'}

def rank_biserial(x, y):
    """Correlação rank-biserial — tamanho de efeito NÃO-PARAMÉTRICO,
    apropriado para Mann-Whitney U (substitui Cohen's d conforme
    solicitado pelo revisor).
        r = 2U/(n1*n2) - 1 ;  |r|: 0.1 pequeno, 0.3 médio, 0.5 grande
    """
    U,_ = mannwhitneyu(x, y, alternative='two-sided')
    return (2*U)/(len(x)*len(y)) - 1

# ── Unidade de análise = SESSÃO (mediana das produções), como no artigo ──
SM = feat.groupby('svd_session')[[c for c in CLIN if c != 'fo_sd']].median()
SM['fo_sd'] = feat[feat.pitch == 'normal'].groupby('svd_session').fo_sd.median()   # fo DP só no pitch normal
SM['label'] = feat.groupby('svd_session').label.first()
SM = SM.reset_index()
print(f'Sessões: {len(SM)} ({(SM.label==0).sum()} saudáveis, {(SM.label==1).sum()} patológicas)')

# ── Análise BINÁRIA em TODAS as features clínicas ─────────────
rows = []
for f in CLIN:
    h = SM.loc[SM.label==0, f].dropna().values
    p = SM.loc[SM.label==1, f].dropna().values
    U,pv = mannwhitneyu(h, p, alternative='two-sided')
    r    = rank_biserial(h, p)
    rows.append({'feature':f,'label':LBL[f],
                 'healthy_median':np.median(h),'healthy_iqr':np.subtract(*np.percentile(h,[75,25])),
                 'path_median':np.median(p),  'path_iqr':np.subtract(*np.percentile(p,[75,25])),
                 'U':U,'p':pv,'p_holm':np.nan,'r_rb':r,'abs_r':abs(r),
                 'magnitude':'large' if abs(r)>=0.5 else 'medium' if abs(r)>=0.3 else 'small'})
bin_df = pd.DataFrame(rows)

# Correção de Holm-Bonferroni (mais potente que Bonferroni simples)
from statsmodels.stats.multitest import multipletests
bin_df['p_holm'] = multipletests(bin_df['p'], method='holm')[1]
bin_df = bin_df.sort_values('abs_r', ascending=False).reset_index(drop=True)

print('═'*100)
print('TABELA 2 — Comparação binária saudável vs. patológico (Mann-Whitney U)')
print('Tamanho de efeito: correlação rank-biserial (não-paramétrica)')
print('═'*100)
print(f"{'Feature':<22}{'Saudável Md(IQR)':>20}{'Patológico Md(IQR)':>21}{'p (Holm)':>12}{'r_rb':>8}{'Magn.':>9}")
print('─'*100)
for _,x in bin_df.iterrows():
    ps = '<0.001' if x.p_holm<0.001 else f'{x.p_holm:.3f}'
    print(f"{x.label:<22}{x.healthy_median:>10.3f} ({x.healthy_iqr:.3f}){x.path_median:>11.3f} ({x.path_iqr:.3f})"
          f"{ps:>12}{x.r_rb:>8.3f}{x.magnitude:>9}")
bin_df.to_csv('/content/table2_binary_effects.csv', index=False)

# ── FIGURA 1 (nova): tamanhos de efeito binários ──────────────
fig, ax = plt.subplots(figsize=(7.0, 4.2))
d = bin_df.sort_values('abs_r')
cols = ['#0072B2' if m=='large' else '#56B4E9' if m=='medium' else '#BBBBBB'
        for m in d.magnitude]
ax.barh(d.label, d.abs_r, color=cols, edgecolor='black', linewidth=0.5, height=0.7)
for th,lb in [(0.1,'pequeno'),(0.3,'médio'),(0.5,'grande')]:
    ax.axvline(th, color='black', ls=':', lw=0.8, alpha=0.7)
    ax.text(th, -0.9, lb, ha='center', fontsize=7.5, style='italic')
for i,(v,pv) in enumerate(zip(d.abs_r, d.p_holm)):
    s = '***' if pv<0.001 else '**' if pv<0.01 else '*' if pv<0.05 else 'ns'
    ax.text(v+0.008, i, f'{v:.3f} {s}', va='center', fontsize=8)
ax.set_xlabel('|r| rank-biserial  (saudável vs. patológico)')
ax.set_xlim(0, max(d.abs_r)*1.28)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', ls='--', alpha=0.4); ax.set_axisbelow(True)
ax.set_title('Discriminação binária por feature acústica Praat\n'
             '(*** p<0.001, ** p<0.01, * p<0.05; correção de Holm)', fontsize=10)
plt.tight_layout()
plt.savefig('/content/Fig1_effect_sizes.tiff'); plt.savefig('/content/Fig1_effect_sizes.png', dpi=300)
plt.show()

# ── TABELA 3: comparação normativa ────────────────────────────
NORMS = {
 'jitter_local_pct':('< 1.040 %','Baken & Orlikoff (2000)'),
 'shimmer_local'   :('< 0.140',  'Baken & Orlikoff (2000)'),
 'shimmer_dB'      :('< 0.350 dB','Baken & Orlikoff (2000)'),
 'hnr_dB'          :('> 20.0 dB','Yumoto et al. (1982)'),
 'cpps_dB'         :('> 14.0 dB','Heman-Ackah et al. (2014)'),
}
print('\n'+'═'*104)
print('TABELA 3 — Valores Praat vs. faixas normativas clínicas publicadas')
print('═'*104)
print(f"{'Feature':<20}{'Saudável (M±DP)':>22}{'Patológico (M±DP)':>22}{'Norma clínica':>16}{'Fonte':>24}")
print('─'*104)
for f,(nm,src) in NORMS.items():
    h = SM.loc[SM.label==0,f].dropna(); p = SM.loc[SM.label==1,f].dropna()
    print(f"{LBL[f]:<20}{h.mean():>11.3f} ± {h.std():<8.3f}{p.mean():>11.3f} ± {p.std():<8.3f}{nm:>16}{src:>24}")
print('✅ Bloco 3 completo')


## Bloco 4 — Classificação com nested CV

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 4 — CLASSIFICAÇÃO BINÁRIA COM NESTED CV
# Responde: R1 "hiperparâmetros sem nested CV", "normalização
#           dentro dos folds", "PCA e leakage"
# ══════════════════════════════════════════════════════════════
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                              f1_score, roc_curve, precision_recall_curve)
import xgboost as xgb
from sklearn.impute import SimpleImputer

X  = feat[FEATS].values
y  = feat['label'].values
g  = feat['svd_speaker'].astype(str).values   # ID real do falante (summary.csv)

outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
inner = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)

# ─── GRADES DE HIPERPARÂMETROS (busca no loop interno) ────────
GRIDS = {
 'SVM-RBF': (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                        ('clf',SVC(kernel='rbf',probability=True,
                                   class_weight='balanced',random_state=RANDOM_STATE))]),
             {'clf__C':[1,10,100], 'clf__gamma':['scale',0.01,0.001]}),
 'Random Forest': (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                              ('clf',RandomForestClassifier(class_weight='balanced',
                                     random_state=RANDOM_STATE, n_jobs=-1))]),
                   {'clf__n_estimators':[300,600], 'clf__max_depth':[None,10,20]}),
 'XGBoost': (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                        ('clf',xgb.XGBClassifier(eval_metric='logloss',verbosity=0,
                               scale_pos_weight=(y==0).sum()/(y==1).sum(),
                               random_state=RANDOM_STATE, n_jobs=-1))]),
             {'clf__n_estimators':[300,600], 'clf__max_depth':[3,4,6],
              'clf__learning_rate':[0.03,0.05,0.1]}),
}

# NOTA METODOLÓGICA (para Methods):
# StandardScaler está DENTRO do Pipeline → os parâmetros de normalização
# (média/DP) são estimados EXCLUSIVAMENTE no fold de treino e aplicados
# ao fold de teste. Idem para PCA no Bloco 6. Nenhuma estatística do
# conjunto de teste participa do ajuste. Isto elimina leakage.

results = {}
best_params_log = {}

print('Nested CV: 5 folds externos × 3 internos (busca de hiperparâmetros)\n')
for name,(pipe,grid) in GRIDS.items():
    res = {'bal_acc':[],'auc':[],'f1':[],'oof_p':np.zeros(len(y)),'oof_y':y.copy()}
    chosen = []
    for k,(tr,te) in enumerate(outer.split(X,y,g)):
        assert not (set(g[tr]) & set(g[te])), 'LEAKAGE DE SPEAKER!'
        gs = GridSearchCV(pipe, grid, scoring='roc_auc',
                          cv=list(inner.split(X[tr],y[tr],g[tr])), n_jobs=-1, refit=True)
        gs.fit(X[tr], y[tr])
        pr = gs.best_estimator_.predict_proba(X[te])[:,1]
        pd_= (pr>=0.5).astype(int)
        res['bal_acc'].append(balanced_accuracy_score(y[te],pd_))
        res['auc'].append(roc_auc_score(y[te],pr))
        res['f1'].append(f1_score(y[te],pd_,average='macro'))
        res['oof_p'][te] = pr
        chosen.append(gs.best_params_)
        print(f'  {name} fold{k+1}: AUC={res["auc"][-1]:.3f} | {gs.best_params_}')
    results[name] = res
    best_params_log[name] = chosen

print('\n'+'═'*62)
print('DESEMPENHO — Nested CV, speaker-independent (binário)')
print('═'*62)
print(f"{'Modelo':<16}{'Bal.Acc':>15}{'AUC-ROC':>15}{'F1 macro':>15}")
print('─'*62)
for n,r in results.items():
    print(f"{n:<16}{np.mean(r['bal_acc']):.3f}±{np.std(r['bal_acc']):.3f}"
          f"{np.mean(r['auc']):>10.3f}±{np.std(r['auc']):.3f}"
          f"{np.mean(r['f1']):>10.3f}±{np.std(r['f1']):.3f}")
json.dump({k:[{kk:str(vv) for kk,vv in p.items()} for p in v]
           for k,v in best_params_log.items()},
          open('/content/best_hyperparams.json','w'), indent=2)
print('✅ Bloco 4 completo')


## Bloco 5 — FT-Transformer

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 5 — FT-TRANSFORMER (nested CV simplificado)
# ══════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import DataLoader, TensorDataset

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

class FTT(nn.Module):
    """Feature Tokenizer Transformer (Gorishniy et al., 2021).
    Cada feature escalar vira um token d-dimensional; [CLS] agrega.
    """
    def __init__(s, nf, d=64, h=8, L=3, p=0.15):
        super().__init__()
        s.W = nn.Parameter(torch.randn(nf,d)*0.02)
        s.b = nn.Parameter(torch.zeros(nf,d))
        s.cls = nn.Parameter(torch.randn(1,1,d)*0.02)
        enc = nn.TransformerEncoderLayer(d, h, d*4, p, batch_first=True, norm_first=True)
        s.enc = nn.TransformerEncoder(enc, L)
        s.norm= nn.LayerNorm(d)
        s.head= nn.Sequential(nn.Linear(d,d//2), nn.GELU(), nn.Dropout(p), nn.Linear(d//2,2))
    def forward(s,x):
        t = x.unsqueeze(-1)*s.W.unsqueeze(0) + s.b.unsqueeze(0)
        t = torch.cat([s.cls.expand(x.shape[0],-1,-1), t], 1)
        return s.head(s.norm(s.enc(t)[:,0]))

def fit_ftt(Xtr,ytr,Xte,yte,Xva=None,yva=None,epochs=60,lr=3e-4,bs=128,d=64,p=0.15):
    imp = SimpleImputer(strategy='median').fit(Xtr)   # imputação ajustada SÓ no treino
    Xtr, Xte = imp.transform(Xtr), imp.transform(Xte)
    sc = StandardScaler().fit(Xtr)          # ajustado SÓ no treino
    Xt, Xe = sc.transform(Xtr), sc.transform(Xte)
    dl = DataLoader(TensorDataset(torch.tensor(Xt,dtype=torch.float32),
                                   torch.tensor(ytr,dtype=torch.long)),
                    batch_size=bs, shuffle=True)
    m  = FTT(Xtr.shape[1], d=d, p=p).to(DEV)
    w  = torch.tensor(1./np.bincount(ytr), dtype=torch.float32).to(DEV)
    crit=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist={'loss':[],'val_auc':[]}
    for ep in range(epochs):
        m.train(); ls=[]
        for xb,yb in dl:
            xb,yb=xb.to(DEV),yb.to(DEV); opt.zero_grad()
            l=crit(m(xb),yb); l.backward()
            nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); ls.append(l.item())
        sch.step()
        m.eval()
        with torch.no_grad():
            pv=Fn.softmax(m(torch.tensor(Xe,dtype=torch.float32).to(DEV)),-1).cpu().numpy()
        hist['loss'].append(np.mean(ls)); hist['val_auc'].append(roc_auc_score(yte,pv[:,1]))
    return pv, hist

ftt={'bal_acc':[],'auc':[],'f1':[],'oof_p':np.zeros(len(y)),'oof_y':y.copy()}
curves=[]
print('Treinando FT-Transformer (5 folds)...')
for k,(tr,te) in enumerate(outer.split(X,y,g)):
    pv,h = fit_ftt(X[tr],y[tr],X[te],y[te])
    pd_  = pv.argmax(1)
    ftt['bal_acc'].append(balanced_accuracy_score(y[te],pd_))
    ftt['auc'].append(roc_auc_score(y[te],pv[:,1]))
    ftt['f1'].append(f1_score(y[te],pd_,average='macro'))
    ftt['oof_p'][te]=pv[:,1]; curves.append(h)
    print(f'  fold{k+1}: AUC={ftt["auc"][-1]:.3f}')
results['FT-Transformer']=ftt
print(f'\nFT-Transformer: AUC={np.mean(ftt["auc"]):.3f}±{np.std(ftt["auc"]):.3f}')
print('✅ Bloco 5 completo')


## Bloco 6 — wav2vec 2.0: congelado e fine-tuned

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 6 — wav2vec 2.0: EMBEDDINGS CONGELADOS **E** FINE-TUNING
# Responde diretamente à crítica central do revisor:
#   "o mismatch de domínio é argumento PARA fine-tuning, não contra"
# Agora avaliamos AMBOS os regimes.
# ══════════════════════════════════════════════════════════════
import torchaudio

bundle = torchaudio.pipelines.WAV2VEC2_BASE
print(f'wav2vec 2.0 BASE | pretraining SR = {bundle.sample_rate} Hz')

# ─── (A) EMBEDDINGS CONGELADOS ────────────────────────────────
EMB_CACHE='/content/w2v_emb.npy'
if Path(EMB_CACHE).exists() and len(np.load(EMB_CACHE))==len(feat):
    E=np.load(EMB_CACHE); print(f'✅ Embeddings em cache: {E.shape}')
else:
    w2v=bundle.get_model().to(DEV).eval()
    out=[]
    for w in tqdm(feat['wav'].values, desc='wav2vec embeddings'):
        try:
            wav,sr=torchaudio.load(w)
            if sr!=bundle.sample_rate:
                wav=torchaudio.functional.resample(wav,sr,bundle.sample_rate)
            wav=wav.mean(0,keepdim=True); wav=wav/(wav.abs().max()+1e-8)
            with torch.no_grad(): fs,_=w2v.extract_features(wav.to(DEV))
            out.append(fs[-1].squeeze(0).mean(0).cpu().numpy())
        except Exception:
            out.append(np.zeros(768,dtype=np.float32))
    E=np.array(out); np.save(EMB_CACHE,E); del w2v; torch.cuda.empty_cache()

# PCA DENTRO do pipeline → ajustado só no treino (sem leakage)
from sklearn.decomposition import PCA
w2v_frozen={'bal_acc':[],'auc':[],'f1':[],'oof_p':np.zeros(len(y)),'oof_y':y.copy()}
pipe_w=Pipeline([('sc',StandardScaler()),('pca',PCA(n_components=50,random_state=RANDOM_STATE)),
                  ('clf',SVC(kernel='rbf',C=10,probability=True,
                             class_weight='balanced',random_state=RANDOM_STATE))])
print('\n(A) Embeddings congelados + SVM:')
for k,(tr,te) in enumerate(outer.split(E,y,g)):
    pipe_w.fit(E[tr],y[tr])
    pr=pipe_w.predict_proba(E[te])[:,1]; pd_=(pr>=0.5).astype(int)
    w2v_frozen['bal_acc'].append(balanced_accuracy_score(y[te],pd_))
    w2v_frozen['auc'].append(roc_auc_score(y[te],pr))
    w2v_frozen['f1'].append(f1_score(y[te],pd_,average='macro'))
    w2v_frozen['oof_p'][te]=pr
    print(f'  fold{k+1}: AUC={w2v_frozen["auc"][-1]:.3f}')
results['wav2vec 2.0 (congelado)']=w2v_frozen

# ─── (B) FINE-TUNING SUPERVISIONADO ───────────────────────────
MAX_S = 3.0   # segundos por gravação
class W2VClf(nn.Module):
    def __init__(s, n_unfreeze=4, p=0.3):
        super().__init__()
        s.bb = bundle.get_model()
        for prm in s.bb.parameters(): prm.requires_grad=False
        # descongelar as últimas n camadas do transformer
        for lay in s.bb.encoder.transformer.layers[-n_unfreeze:]:
            for prm in lay.parameters(): prm.requires_grad=True
        s.head=nn.Sequential(nn.LayerNorm(768), nn.Dropout(p), nn.Linear(768,2))
    def forward(s,w):
        f,_=s.bb.extract_features(w)
        return s.head(f[-1].mean(1))

def load_batch(paths, sr=16000, maxs=MAX_S):
    L=int(sr*maxs); out=[]
    for p_ in paths:
        w,s_=torchaudio.load(p_)
        if s_!=sr: w=torchaudio.functional.resample(w,s_,sr)
        w=w.mean(0); w=w/(w.abs().max()+1e-8)
        w=w[:L] if len(w)>=L else torch.nn.functional.pad(w,(0,L-len(w)))
        out.append(w)
    return torch.stack(out)

def finetune(tr_idx, te_idx, epochs=3, bs=8, lr=1e-5):
    m=W2VClf().to(DEV)
    opt=torch.optim.AdamW([p_ for p_ in m.parameters() if p_.requires_grad],
                           lr=lr, weight_decay=0.01)
    w=torch.tensor(1./np.bincount(y[tr_idx]),dtype=torch.float32).to(DEV)
    crit=nn.CrossEntropyLoss(weight=w)
    paths=feat['wav'].values
    for ep in range(epochs):
        m.train(); perm=np.random.permutation(tr_idx)
        for i in range(0,len(perm),bs):
            b=perm[i:i+bs]
            xb=load_batch(paths[b]).to(DEV); yb=torch.tensor(y[b]).to(DEV)
            opt.zero_grad(); l=crit(m(xb),yb); l.backward()
            nn.utils.clip_grad_norm_([p_ for p_ in m.parameters() if p_.requires_grad],1.0)
            opt.step()
    m.eval(); prs=[]
    with torch.no_grad():
        for i in range(0,len(te_idx),bs):
            b=te_idx[i:i+bs]
            xb=load_batch(paths[b]).to(DEV)
            prs.append(Fn.softmax(m(xb),-1)[:,1].cpu().numpy())
    del m; torch.cuda.empty_cache()
    return np.concatenate(prs)

RUN_FT = True   # ⚠️ requer GPU; ~40-60 min
if RUN_FT:
    w2v_ft={'bal_acc':[],'auc':[],'f1':[],'oof_p':np.zeros(len(y)),'oof_y':y.copy()}
    print('\n(B) Fine-tuning (últimas 4 camadas, lr=1e-5, 3 épocas):')
    for k,(tr,te) in enumerate(outer.split(X,y,g)):
        pr=finetune(tr,te)
        pd_=(pr>=0.5).astype(int)
        w2v_ft['bal_acc'].append(balanced_accuracy_score(y[te],pd_))
        w2v_ft['auc'].append(roc_auc_score(y[te],pr))
        w2v_ft['f1'].append(f1_score(y[te],pd_,average='macro'))
        w2v_ft['oof_p'][te]=pr
        print(f'  fold{k+1}: AUC={w2v_ft["auc"][-1]:.3f}')
    results['wav2vec 2.0 (fine-tuned)']=w2v_ft
    print(f'\nFine-tuned: AUC={np.mean(w2v_ft["auc"]):.3f}±{np.std(w2v_ft["auc"]):.3f}')
print('✅ Bloco 6 completo')


## Bloco 7 — Comparação, SHAP e limiar clínico

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCO 7 — COMPARAÇÃO, INTERPRETABILIDADE E LIMIAR CLÍNICO
# Responde: R1 "ROCs em painel único", "coluna de parâmetros",
#           "interpretar todas as figuras", "explicar Youden J"
# ══════════════════════════════════════════════════════════════
from itertools import combinations
import shap

# ── TABELA 4 ──────────────────────────────────────────────────
NPAR = {  # nº de parâmetros ajustados — revisor apontou que "—" é incorreto
 'SVM-RBF':f'≈ n vetores de suporte × {len(FEATS)}',
 'Random Forest':'300–600 árvores',
 'XGBoost':'300–600 árvores',
 'FT-Transformer':'161.506',
 'wav2vec 2.0 (congelado)':'0 treináveis (94,7 M congelados)',
 'wav2vec 2.0 (fine-tuned)':'≈ 28,4 M treináveis (de 94,7 M)',
}
print('═'*94)
print('TABELA 4 — Desempenho de classificação binária (nested CV, speaker-independent)')
print('═'*94)
print(f"{'Modelo':<26}{'Bal.Acc':>14}{'AUC-ROC':>14}{'F1 macro':>14}{'Parâmetros':>26}")
print('─'*94)
for n,r in results.items():
    print(f"{n:<26}{np.mean(r['bal_acc']):.3f}±{np.std(r['bal_acc']):.3f}"
          f"{np.mean(r['auc']):>9.3f}±{np.std(r['auc']):.3f}"
          f"{np.mean(r['f1']):>9.3f}±{np.std(r['f1']):.3f}{NPAR.get(n,'—'):>26}")

# ── TABELA 5 — bootstrap IC95% pareado ────────────────────────
rng=np.random.default_rng(RANDOM_STATE)
print('\n'+'═'*88)
print('TABELA 5 — Diferenças de AUC entre modelos (bootstrap IC95%, 2000 reamostragens)')
print('═'*88)
print(f"{'Modelo A':<26}{'Modelo B':<26}{'ΔAUC':>9}{'IC95%':>20}{'Sig':>6}")
print('─'*88)
boot_rows=[]
# Bootstrap PAREADO de falantes sobre as previsões fora da amostra (todas as gravações do falante sorteado entram juntas)
_spk = np.unique(g); _rows = pd.Series(np.arange(len(y))).groupby(g).apply(np.array)
_B = {n: [] for n in results}
for _ in range(2000):
    idx = np.concatenate(_rows.loc[rng.choice(_spk, len(_spk))].values)
    for n in results:
        _B[n].append(roc_auc_score(y[idx], results[n]['oof_p'][idx]))
_B = {n: np.array(v) for n, v in _B.items()}
for a,b in combinations(results.keys(),2):
    dl = _B[a] - _B[b]
    lo,hi=np.percentile(dl,[2.5,97.5])
    dd = roc_auc_score(y, results[a]['oof_p']) - roc_auc_score(y, results[b]['oof_p'])
    sig='sim' if lo*hi>0 else 'não'
    print(f"{a:<26}{b:<26}{dd:>+9.4f}{f'[{lo:+.3f}, {hi:+.3f}]':>20}{sig:>6}")
    boot_rows.append({'A':a,'B':b,'delta':dd,'lo':lo,'hi':hi,'sig':sig})
pd.DataFrame(boot_rows).to_csv('/content/table5_bootstrap.csv',index=False)

# ── FIGURA 2 — ROCs em PAINEL ÚNICO (pedido do revisor) ───────
fig,ax=plt.subplots(figsize=(5.6,5.2))
mk=['-','--','-.',':','-','--']
for i,(n,r) in enumerate(results.items()):
    fpr,tpr,_=roc_curve(r['oof_y'],r['oof_p'])
    ax.plot(fpr,tpr,mk[i%len(mk)],color=PAL[i%len(PAL)],lw=1.8,
            label=f"{n} (AUC={roc_auc_score(r['oof_y'],r['oof_p']):.3f})")
ax.plot([0,1],[0,1],color='black',lw=0.8,ls=':',label='Acaso')
ax.set_xlabel('1 − Especificidade'); ax.set_ylabel('Sensibilidade')
ax.set_title('Curvas ROC out-of-fold — todos os modelos', fontsize=10)
ax.legend(loc='lower right', frameon=False, fontsize=8)
ax.spines[['top','right']].set_visible(False); ax.grid(ls='--',alpha=0.4)
ax.set_aspect('equal'); plt.tight_layout()
plt.savefig('/content/Fig2_ROC_panel.tiff'); plt.savefig('/content/Fig2_ROC_panel.png',dpi=300)
plt.show()

# ── FIGURA 3 — SHAP: o modelo aprendeu o sinal clínico? ───────
# ESTA É A PONTE entre estatística e ML que o revisor cobrou.
best = max(results, key=lambda k: np.mean(results[k]['auc']) if 'wav2vec' not in k else -1)
sp=np.random.default_rng(RANDOM_STATE)
uspk=feat.svd_speaker.unique()
trs=sp.choice(uspk,int(len(uspk)*0.8),replace=False)
mtr=feat.svd_speaker.isin(trs).values
scS=StandardScaler().fit(X[mtr])
mdl=xgb.XGBClassifier(n_estimators=600,max_depth=4,learning_rate=0.05,
      eval_metric='logloss',verbosity=0,random_state=RANDOM_STATE,
      scale_pos_weight=(y[mtr]==0).sum()/(y[mtr]==1).sum()).fit(scS.transform(X[mtr]),y[mtr])
expl=shap.TreeExplainer(mdl); SV=expl.shap_values(scS.transform(X[~mtr]))
if not isinstance(SV,np.ndarray): SV=SV[1]

GRP={'Jitter':[f for f in FEATS if 'jitter' in f],
     'Shimmer':[f for f in FEATS if 'shimmer' in f],
     'HNR / CPPS':[f for f in FEATS if f in ('hnr_dB','cpps_dB')],
     'fo / sonorização':[f for f in FEATS if f.startswith('fo_') or 'voiced' in f],
     'Espectrais':[f for f in FEATS if f.startswith('spec_') or f.startswith('rms') or f.startswith('zcr')],
     'MFCC':[f for f in FEATS if 'mfcc' in f]}
gi={k:np.abs(SV[:,[FEATS.index(f) for f in v if f in FEATS]]).mean()
    for k,v in GRP.items() if any(f in FEATS for f in v)}

fig,axs=plt.subplots(1,2,figsize=(11,4.4))
srt=dict(sorted(gi.items(),key=lambda x:x[1]))
axs[0].barh(list(srt),list(srt.values()),color=PAL[:len(srt)],
            edgecolor='black',lw=0.5,height=0.7)
axs[0].set_xlabel('|SHAP| médio'); axs[0].set_title('(A) Contribuição por família acústica',fontsize=10)
axs[0].spines[['top','right']].set_visible(False); axs[0].grid(axis='x',ls='--',alpha=0.4)
ms=np.abs(SV).mean(0); t15=np.argsort(ms)[-15:]
cl=[next((PAL[list(GRP).index(k)%len(PAL)] for k,v in GRP.items() if FEATS[i] in v),'#999')
    for i in t15]
axs[1].barh([FEATS[i] for i in t15],ms[t15],color=cl,edgecolor='black',lw=0.5,height=0.7)
axs[1].set_xlabel('|SHAP| médio'); axs[1].set_title('(B) 15 features mais influentes',fontsize=10)
axs[1].spines[['top','right']].set_visible(False); axs[1].grid(axis='x',ls='--',alpha=0.4)
plt.tight_layout()
plt.savefig('/content/Fig3_SHAP.tiff'); plt.savefig('/content/Fig3_SHAP.png',dpi=300)
plt.show()

# Concordância estatística × modelo (a INTEGRAÇÃO explícita)
print('\n'+'═'*74)
print('INTEGRAÇÃO: ranking estatístico (r rank-biserial) × ranking SHAP')
print('═'*74)
print(f"{'Feature':<22}{'r_rb (Tab.2)':>14}{'rank estat.':>13}{'|SHAP|':>11}{'rank SHAP':>12}")
print('─'*74)
shap_rank={FEATS[i]:r+1 for r,i in enumerate(np.argsort(-ms))}
for j,row in bin_df.head(8).iterrows():
    f=row.feature
    print(f"{LBL[f]:<22}{row.r_rb:>14.3f}{j+1:>13}"
          f"{ms[FEATS.index(f)]:>11.4f}{shap_rank.get(f,'—'):>12}")
from scipy.stats import spearmanr
common=[f for f in CLIN if f in FEATS]
rho,pv_=spearmanr([bin_df.set_index('feature').loc[f,'abs_r'] for f in common],
                  [ms[FEATS.index(f)] for f in common])
print(f"\nCorrelação de Spearman entre os dois rankings: ρ = {rho:.3f} (p = {pv_:.4f})")

# ── FIGURA 4 — limiar clínico OOF ─────────────────────────────
r=results[best]; fpr,tpr,thr=roc_curve(r['oof_y'],r['oof_p'])
J=np.argmax(tpr-fpr); pr_,rc_,_=precision_recall_curve(r['oof_y'],r['oof_p'])
fig,axs=plt.subplots(1,2,figsize=(10,4.4))
axs[0].plot(fpr,tpr,color=PAL[0],lw=1.8)
axs[0].scatter(fpr[J],tpr[J],color=PAL[1],s=70,zorder=5,
   label=f"J de Youden\nlimiar={thr[J]:.3f}\nSens={tpr[J]:.3f}\nEsp={1-fpr[J]:.3f}")
axs[0].plot([0,1],[0,1],color='black',ls=':',lw=0.8)
axs[0].set_xlabel('1 − Especificidade'); axs[0].set_ylabel('Sensibilidade')
axs[0].set_title(f'(A) ROC out-of-fold — {best}',fontsize=10)
axs[0].legend(frameon=False,fontsize=8,loc='lower right')
axs[0].spines[['top','right']].set_visible(False); axs[0].grid(ls='--',alpha=0.4)
axs[1].plot(rc_,pr_,color=PAL[2],lw=1.8)
axs[1].axhline((r['oof_y']==1).mean(),color='black',ls=':',lw=0.8,label='Prevalência')
axs[1].set_xlabel('Sensibilidade (recall)'); axs[1].set_ylabel('Valor preditivo positivo')
axs[1].set_title('(B) Curva precisão–sensibilidade',fontsize=10)
axs[1].legend(frameon=False,fontsize=8)
axs[1].spines[['top','right']].set_visible(False); axs[1].grid(ls='--',alpha=0.4)
plt.tight_layout()
plt.savefig('/content/Fig4_threshold.tiff'); plt.savefig('/content/Fig4_threshold.png',dpi=300)
plt.show()

sens,spec=tpr[J],1-fpr[J]; prev=(r['oof_y']==1).mean()
ppv=sens*prev/(sens*prev+(1-spec)*(1-prev)); npv=spec*(1-prev)/((1-sens)*prev+spec*(1-prev))
print(f"""
LIMIAR CLÍNICO (J de Youden sobre {len(r['oof_y'])} predições out-of-fold)
  J de Youden = max(sensibilidade + especificidade − 1); identifica o ponto
  da curva ROC mais distante da diagonal do acaso.
  Limiar ......... {thr[J]:.3f}
  Sensibilidade .. {sens:.3f}
  Especificidade . {spec:.3f}
  VPP ............ {ppv:.3f}   (prevalência = {prev:.3f})
  VPN ............ {npv:.3f}
""")
print('✅ Bloco 7 completo — figuras em TIFF 600 dpi')
